# ENEC 2026 · Day 4 · The A-001 Review App, Running Inside Colab

**Time:** about 45 minutes  |  **Runs in:** Google Colab (free tier is fine)  |  **Needs:** OpenAI key in Colab Secrets (`OPENAI_API_KEY`)

**Data:** a small slice of the synthetic *Nuclear Enterprise 360* dataset, focused on **Asset A-001** (a cooling-water pump). All data is synthetic training data.

**Goal:** Wrap the capstone into a small web app (Gradio) that runs inside this Colab session. No separate hosting or deployment platform is needed.

> Training notice: nothing in this notebook is an engineering diagnosis or a maintenance authorisation. The AI supports a qualified human reviewer.

In [ ]:
# Install what Colab does not already have (about 20 seconds)
%pip install -q -U openai gradio

In [ ]:
# Get the course files (data, documents, skills) and the helper module.
# In Colab this clones the public course repo into /content/enec2026oct.
import os, sys, subprocess

REPO_URL = "https://github.com/Decoding-Data-Science/enec2026oct"
IN_COLAB = os.path.isdir("/content") and "google.colab" in sys.modules
ROOT = "/content/enec2026oct" if IN_COLAB else os.getcwd()

if IN_COLAB and not os.path.isdir(ROOT):
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, ROOT], capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit(
            "Could not clone the course repo:\n" + r.stderr[-300:] +
            "\nFallback: upload enec2026oct.zip in the Files panel (left sidebar), "
            "run  !unzip -q enec2026oct.zip -d /content  and run this cell again."
        )

sys.path.insert(0, os.path.join(ROOT, "colab"))
import enec_colab as enec
print("Course files ready at:", enec.ROOT)

In [ ]:
# Connect to OpenAI. The key comes from Colab Secrets and is never printed.
client = enec.get_client()
MODEL = enec.pick_model(client)     # first available of: gpt-5.4-mini, gpt-5-mini, gpt-4o-mini

## What you are building

A three-tab app on top of everything from Days 1-4:

| Tab | What it does |
|---|---|
| **Ask A-001** | The tool-calling agent from Day 3 (SQL + documents + forecast) with a visible tool trace |
| **Review briefing** | The Day 4 capstone: evidence packet, 8-section briefing and automatic checks |
| **Document register** | Which documents are APPROVED, DRAFT or SUPERSEDED |

A banner on every tab states the boundary: *decision support only, a qualified human decides.*

**Where it runs:** inside this Colab session. It stops when the session stops.

In [ ]:
import json, pandas as pd, gradio as gr

tables = enec.load_tables()
con    = enec.make_db(tables)
docs   = enec.load_documents(include_day4=True)
index  = enec.RAGIndex.build(client, docs)
hourly = tables["a001_sensor_hourly_90d"]
TOOLS, IMPLS = enec.build_a001_tools(con, index, hourly)

SYSTEM = f"""You are a reliability-review assistant for synthetic asset A-001.
Use sql_query for database facts, search_documents for procedures, policies and reports, forecast for the vibration trend.
{enec.SCHEMA_HINT}
Rules: only APPROVED documents are current authority; DRAFT is not authoritative; SUPERSEDED is historical only.
Cite document IDs in [brackets] and table names for database facts. Separate DATABASE, DOCUMENT and FORECAST evidence.
If evidence is missing, say so. You never authorise maintenance, shutdown or replacement; a qualified human decides."""
BANNER = "**Decision support only.** Synthetic training data. A qualified human reviewer owns every decision."
print("ready")

## The three functions behind the tabs

In [ ]:
def ask_agent(question):
    if not question or not question.strip():
        return "Please type a question.", pd.DataFrame()
    final, trace, _ = enec.run_agent(client, question, SYSTEM, TOOLS, IMPLS, verbose=False)
    t = pd.DataFrame(trace)[["step", "tool", "status", "args"]] if trace else pd.DataFrame(columns=["step", "tool", "status", "args"])
    t["args"] = t["args"].astype(str).str.slice(0, 160)
    return final, t

def make_briefing():
    packet = enec.build_evidence_packet(con, index, hourly, docs)
    text = enec.generate_briefing(client, packet)
    return text, enec.check_briefing(text, docs)

def register():
    return pd.DataFrame([{k: d[k] for k in ("doc_id", "title", "status", "version", "date")} for d in docs]).sort_values(["status", "doc_id"])

# Test them here first, then wrap them in a UI
ans, trace = ask_agent("How many high-priority work orders are open for A-001?")
print(ans); trace

## The user interface

`gr.Blocks` lays out components; each button calls one of the functions above.

In [ ]:
EXAMPLES = ["Which procedure is currently approved for A-001, and is the draft in force?",
            "How many work orders are open for A-001 and how many are high priority?",
            "Is A-001 vibration likely to keep rising over the next week? How reliable is that forecast?",
            "Prepare A-001 for human reliability review."]

with gr.Blocks(title="A-001 Review Assistant") as demo:
    gr.Markdown("# A-001 Review Assistant\n" + BANNER)
    with gr.Tab("Ask A-001"):
        q = gr.Textbox(label="Your question", lines=2)
        gr.Examples(EXAMPLES, inputs=q)
        go = gr.Button("Ask", variant="primary")
        out = gr.Markdown()
        trace_df = gr.Dataframe(label="Tool trace (what the agent actually did)", wrap=True)
        go.click(ask_agent, q, [out, trace_df])
    with gr.Tab("Review briefing"):
        gr.Markdown("Builds the evidence packet, writes the 8-section briefing and runs the automatic checks (about 20-40 seconds).")
        b = gr.Button("Generate briefing", variant="primary")
        brief = gr.Markdown()
        checks = gr.Dataframe(label="Automatic checks", wrap=True)
        b.click(make_briefing, None, [brief, checks])
    with gr.Tab("Document register"):
        gr.Markdown("Only **APPROVED** documents are current authority.")
        reg = gr.Dataframe(value=register(), wrap=True)
print("UI built")

## Launch

- `SHARE = False` (default): the app opens inside the notebook output. Safest.
- `SHARE = True`: Gradio creates a temporary public link so a room can open it on their own devices. **Anyone with the link can use your OpenAI key through the app.** Use it only for a live demo, then stop the cell.

In [ ]:
SHARE = False
demo.launch(share=SHARE, debug=False)

## Before you show it to anyone

- [ ] Run each example question once and read the **tool trace**
- [ ] Generate the briefing once and read the checks
- [ ] Check your OpenAI usage dashboard for cost
- [ ] If you used `SHARE = True`, stop the cell when finished (Runtime menu, then Interrupt)

## Stop the app

Run `demo.close()` or stop the cell.

In [ ]:
# demo.close()